In [2]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import cvxpy as cp

In [8]:
from scipy.special import eval_hermitenorm

In [2]:
def create_dataset(theta, n_points, n_correlated_dimensions, n_uncorrelated_dimensions, noise = 0):
    n_dim = n_correlated_dimensions + n_uncorrelated_dimensions

    # Generate transition matrix
    T = np.zeros((n_dim, n_dim))
    np.fill_diagonal(T, np.cos(theta))

    for i in range(n_correlated_dimensions - 1):
        T[i, i+1] = np.sin(theta)

    for i in range(n_correlated_dimensions - 1):
        T[i+1, i] = -np.sin(theta)

    T[1, 2] = 0 # Decouple first two dimensions from rest

    # Generate data
    xi = np.random.random(n_dim)
    x = [xi]
    for i in range(n_points):
        xi = T@xi + np.random.normal(0, noise, size=(n_dim))
        x.append(xi)

    return np.stack(x)

In [ ]:
class HermiteBasis:
    def __init__(self, bandwidth):
        self.bandwidth = bandwidth

    def __call__(self, x):
        return 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy
    
class Rascutti:
    def __init__(self, bandwidth, lam=0, rho=0, eig_bw=0.05, mercer=Mercer, eigen_K = 10):
        self.mercer = mercer(bandwidth=bandwidth, eig_bw=eig_bw, n_samples=30)
        self.eigen_K = eigen_K
        self.lam = lam
        self.rho = rho

    def _fit_rascutti(self, X, y, eigenvalues, eigenvectors, eigen_K, lam, rho):
        N, d = X.shape

        Q_mercer_k = eigenvectors[:, :, :eigen_K]
        Q_k = Q_mercer_k 
        D = np.diag(eigenvalues[:eigen_K])

        ## Square root matrices
        Dinvsqroot = np.sqrt(np.linalg.inv(D))
        betas = [cp.Variable(eigen_K) for j in range(d)]
        t = cp.Variable()
        u = [cp.Variable() for j in range(d)]
        v = [cp.Variable() for j in range(d)]

        Q_sum = cp.sum([Q_k[j, :, :]@betas[j] for j in range(d)], axis=0)

        socp_constraints = [
            *[cp.SOC(1/np.sqrt(2) * (0.5 + 1), cp.hstack([1/np.sqrt(2) * (0.5 - 1), Dinvsqroot@betas[j]])) for j in range(d)],
            *[cp.SOC(v[j], betas[j]) for j in range(d)],
            *[cp.SOC(u[j], Dinvsqroot@betas[j]) for j in range(d)],
            cp.SOC(np.sqrt(2)*(t + 0.5)/2, cp.hstack([np.sqrt(2)*(t - 0.5)/2, y - Q_sum]))
        ]

        prob = cp.Problem(
            cp.Minimize(1/(2*N)*t+ (lam/np.sqrt(N))*cp.sum(v) + rho*cp.sum(u)),
            socp_constraints
        )

        prob.solve()

        return betas


    def fit(self, X, Y):
        N, d= X.shape

        self.y_shift = np.mean(Y, axis=0)

        y_centered = Y - self.y_shift

        self.mercer.fit(X)
        eigenvalues = self.mercer.eigenvalues*N
        eigenvectors = self.mercer.predict(X)/np.sqrt(N)

        betas = [
            self._fit_rascutti(
                X, 
                y_centered[:, i], 
                eigenvalues=eigenvalues, 
                eigenvectors=eigenvectors,
                eigen_K=self.eigen_K,
                lam = self.lam,
                rho = self.rho
            ) for i in range(Y.shape[1])
        ]

        self.beta = np.stack([np.vstack([b.value for b in betas[j]]).T for j in range(len(betas))])

    def predict(self, X):
        N, d= X.shape
        eigenvectors = self.mercer.predict(X)/np.sqrt(N)
        Q_k = eigenvectors[:, :, :self.eigen_K]
        fits = np.einsum("vdf,tfv->dt", Q_k, self.beta) + self.y_shift
        return fits